# Chapter 15 --- Statistical Process Control and Release Gates

**Monograph:** *How to Build Policy-Governed AI Agents*

**Prerequisites:**
- Chapter 8 notebook completed (ClaimVerifier).
- Chapter 12 concepts (PolicyEngine).
- Chapter 13 concepts (BehavioralContracts).
- Chapter 14 concepts (Telemetry and Tracing).

**What you will build:**
- `governed_agent/release_gates.py` --- TieredReleaseGate, RunMetrics, RunTrend, GateThresholds, GateResult.

**Key idea:** Deployment decisions should be gated by deterministic, monotonically ordered thresholds. Statistical process control detects drift before it causes harm.

## Preamble

### Key Concepts

- **Release gate:** A deterministic checkpoint that compares run-level metrics against
  pre-defined thresholds before allowing deployment. Replaces (or supplements) human
  sign-off in the model risk management process.
- **Deployment tiers:** Three levels of increasing autonomy:
  - *Advisory* --- agent output shown alongside a human's own work (low bar).
  - *Recommendation* --- agent output presented as a formal recommendation (moderate bar).
  - *Action-taking* --- agent acts autonomously (strict bar, including fairness).
- **Monotonicity invariant:** If a run passes a stricter tier it must also pass all
  looser tiers. This is a safety property, not a design goal.
- **RunMetrics fields:**
  - *accuracy* --- test-set prediction accuracy.
  - *confidence* --- mean predicted-class probability (model certainty proxy).
  - *robustness_drop* --- accuracy decrease under feature perturbation (lower is better).
  - *fairness_air* --- adverse impact ratio (four-fifths rule: >= 0.80).
  - *tool_calls* --- number of tool invocations in the run.
  - *policy_blocks* --- number of policy violations detected.
- **Statistical process control (SPC):** A technique from manufacturing that monitors
  quality metrics over time and fires an alert when the trend turns negative.
- **OLS slope:** Ordinary Least Squares regression slope --- the rate of change of
  a metric across runs. Negative slope = degradation.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print(f"NumPy: {np.__version__}")
print("Ready.")

## Import Release Gate Components

We import the four dataclasses and the `TieredReleaseGate` class built in `governed_agent/release_gates.py`.

In [ ]:
from knowlytix.harness.governance.release_gates import (
    RunMetrics, RunTrend,
    TieredReleaseGate,
)

gate = TieredReleaseGate()
print("Tier thresholds:")
for tier_name in gate.TIER_ORDER:
    t = gate.TIERS[tier_name]
    print(f"  {tier_name:18s}  acc>={t.min_accuracy:.2f}  "
          f"conf>={t.min_confidence:.2f}  "
          f"rob<={t.max_robustness_drop:.2f}  "
          f"air>={t.min_fairness_air:.2f}")

---
## Lab 15.1 --- Build Tiered Gates and Verify Monotonicity

We generate 10 synthetic `RunMetrics` with progressively degrading quality (from excellent to poor) and check each against all three tiers. The monotonicity property --- advisory passes >= recommendation passes >= action_taking passes --- must hold for every single run.

### Generate 10 Synthetic RunMetrics

Each run degrades linearly: accuracy drops from 0.95 to 0.50, confidence from 0.85 to 0.15, robustness_drop grows from 0.01 to 0.20 and fairness_air drops from 0.95 to 0.55.

In [3]:
rng = np.random.RandomState(42)
n_runs = 10

synthetic_metrics = []
for i in range(n_runs):
    frac = i / (n_runs - 1)  # 0.0 (best) to 1.0 (worst)
    m = RunMetrics(
        accuracy=round(0.95 - 0.45 * frac, 4),
        confidence=round(0.85 - 0.70 * frac, 4),
        robustness_drop=round(0.01 + 0.19 * frac, 4),
        fairness_air=round(0.95 - 0.40 * frac, 4),
        tool_calls=int(8 + rng.randint(0, 5)),
        policy_blocks=int(rng.randint(0, 3)),
    )
    synthetic_metrics.append(m)

print(f"Generated {len(synthetic_metrics)} synthetic runs.\n")
print(f"{'Run':<5s} {'Acc':>7s} {'Conf':>7s} {'RobDrop':>8s} {'AIR':>7s} "
      f"{'Tools':>6s} {'Blocks':>7s}")
print("-" * 55)
for i, m in enumerate(synthetic_metrics):
    print(f"{i:<5d} {m.accuracy:7.4f} {m.confidence:7.4f} "
          f"{m.robustness_drop:8.4f} {m.fairness_air:7.4f} "
          f"{m.tool_calls:6d} {m.policy_blocks:7d}")

Generated 10 synthetic runs.

Run       Acc    Conf  RobDrop     AIR  Tools  Blocks
-------------------------------------------------------
0      0.9500  0.8500   0.0100  0.9500     11       0
1      0.9000  0.7722   0.0311  0.9056     10       0
2      0.8500  0.6944   0.0522  0.8611     12       2
3      0.8000  0.6167   0.0733  0.8167      9       2
4      0.7500  0.5389   0.0944  0.7722     10       2
5      0.7000  0.4611   0.1156  0.7278     12       2
6      0.6500  0.3833   0.1367  0.6833     12       1
7      0.6000  0.3056   0.1578  0.6389     11       1
8      0.5500  0.2278   0.1789  0.5944      9       0
9      0.5000  0.1500   0.2000  0.5500      8       1


### Check All Tiers and Build Summary Table

For each of the 10 runs, check against all three tiers. Display a summary table showing which tiers pass for each run.

In [4]:
all_results = []

print(f"{'Run':<5s} {'Accuracy':>9s} {'Advisory':>10s} {'Recommend':>10s} "
      f"{'Action':>10s} {'Monotonic':>10s}")
print("-" * 60)

tier_pass_counts = {"advisory": 0, "recommendation": 0, "action_taking": 0}

for i, m in enumerate(synthetic_metrics):
    results = gate.check_all_tiers(m)
    all_results.append(results)
    mono = gate.verify_monotonicity(m)

    for t in gate.TIER_ORDER:
        if results[t].passed:
            tier_pass_counts[t] += 1

    adv = "PASS" if results["advisory"].passed else "FAIL"
    rec = "PASS" if results["recommendation"].passed else "FAIL"
    act = "PASS" if results["action_taking"].passed else "FAIL"
    mono_str = "YES" if mono else "** NO **"

    print(f"{i:<5d} {m.accuracy:9.4f} {adv:>10s} {rec:>10s} "
          f"{act:>10s} {mono_str:>10s}")

print(f"\n=== Tier Pass Counts (out of {n_runs}) ===")
for t in gate.TIER_ORDER:
    print(f"  {t:18s}: {tier_pass_counts[t]:2d}/{n_runs}")

# Verify monotonicity for ALL runs
all_mono = all(gate.verify_monotonicity(m) for m in synthetic_metrics)
print(f"\nMonotonicity holds for all {n_runs} runs: {all_mono}")
assert all_mono, "Monotonicity violation detected!"
print("Monotonicity property VERIFIED.")

Run    Accuracy   Advisory  Recommend     Action  Monotonic
------------------------------------------------------------
0        0.9500       PASS       PASS       PASS        YES
1        0.9000       PASS       PASS       PASS        YES
2        0.8500       PASS       PASS       FAIL        YES
3        0.8000       PASS       PASS       FAIL        YES
4        0.7500       PASS       FAIL       FAIL        YES
5        0.7000       PASS       FAIL       FAIL        YES
6        0.6500       FAIL       FAIL       FAIL        YES
7        0.6000       FAIL       FAIL       FAIL        YES
8        0.5500       FAIL       FAIL       FAIL        YES
9        0.5000       FAIL       FAIL       FAIL        YES

=== Tier Pass Counts (out of 10) ===
  advisory          :  6/10
  recommendation    :  4/10
  action_taking     :  2/10

Monotonicity holds for all 10 runs: True
Monotonicity property VERIFIED.


### Detailed Gate Result for a Borderline Run

Show the full gate result for run 4 (near the recommendation/action_taking boundary) to illustrate how blocking reasons are reported.

In [5]:
borderline_idx = 4
m_border = synthetic_metrics[borderline_idx]
print(f"=== Detailed Gate Result for Run {borderline_idx} ===")
print(f"  accuracy={m_border.accuracy}, confidence={m_border.confidence}, "
      f"robustness_drop={m_border.robustness_drop}, "
      f"fairness_air={m_border.fairness_air}\n")

for tier in gate.TIER_ORDER:
    result = all_results[borderline_idx][tier]
    print(gate.format_gate_result(result))
    print()

=== Detailed Gate Result for Run 4 ===
  accuracy=0.75, confidence=0.5389, robustness_drop=0.0944, fairness_air=0.7722

  [PASS] advisory
    [+] accuracy: 0.7500 (threshold: 0.70)
    [+] confidence: 0.5389 (threshold: 0.30)
    [+] robustness_drop: 0.0944 (threshold: 1.00)
    [+] fairness_air: 0.7722 (threshold: 0.00)

  [FAIL] recommendation
    [X] accuracy: 0.7500 (threshold: 0.80)
    [+] confidence: 0.5389 (threshold: 0.50)
    [+] robustness_drop: 0.0944 (threshold: 0.10)
    [+] fairness_air: 0.7722 (threshold: 0.00)
    Blocking: accuracy 0.7500 < 0.80

  [FAIL] action_taking
    [X] accuracy: 0.7500 (threshold: 0.85)
    [X] confidence: 0.5389 (threshold: 0.70)
    [X] robustness_drop: 0.0944 (threshold: 0.05)
    [X] fairness_air: 0.7722 (threshold: 0.80)
    Blocking: accuracy 0.7500 < 0.85; confidence 0.5389 < 0.70; robustness_drop 0.0944 > 0.05; fairness_air 0.7722 < 0.80



### Visualize Gate Results

Plot a heatmap showing pass/fail across runs (x-axis) and tiers (y-axis). Green = pass, red = fail.

In [6]:
# Build pass/fail matrix: rows = tiers, columns = runs
matrix = np.zeros((3, n_runs))
for j, m in enumerate(synthetic_metrics):
    results = all_results[j]
    for i, tier in enumerate(gate.TIER_ORDER):
        matrix[i, j] = 1.0 if results[tier].passed else 0.0

fig, axes = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={"height_ratios": [1, 2]})

# Top panel: heatmap
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#e74c3c", "#2ecc71"])
im = axes[0].imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=1)
axes[0].set_yticks(range(3))
axes[0].set_yticklabels(["Advisory", "Recommendation", "Action-Taking"])
axes[0].set_xticks(range(n_runs))
axes[0].set_xticklabels([f"R{i}" for i in range(n_runs)])
axes[0].set_xlabel("Run (best to worst)")
axes[0].set_title("Release Gate Results: Green = PASS, Red = FAIL")

# Annotate cells
for i in range(3):
    for j in range(n_runs):
        label = "P" if matrix[i, j] == 1.0 else "F"
        color = "white" if matrix[i, j] == 0 else "black"
        axes[0].text(j, i, label, ha="center", va="center",
                     fontsize=9, fontweight="bold", color=color)

# Bottom panel: metric traces
accuracies = [m.accuracy for m in synthetic_metrics]
confidences = [m.confidence for m in synthetic_metrics]
rob_drops = [m.robustness_drop for m in synthetic_metrics]
air_vals = [m.fairness_air for m in synthetic_metrics]
x = range(n_runs)

axes[1].plot(x, accuracies, "o-", label="Accuracy", color="#2980b9", linewidth=2)
axes[1].plot(x, confidences, "s-", label="Confidence", color="#27ae60", linewidth=2)
axes[1].plot(x, rob_drops, "^-", label="Robustness Drop", color="#e74c3c", linewidth=2)
axes[1].plot(x, air_vals, "D-", label="Fairness AIR", color="#8e44ad", linewidth=2)

# Threshold lines
axes[1].axhline(0.70, color="#2980b9", linestyle="--", alpha=0.4, label="Adv. acc threshold")
axes[1].axhline(0.85, color="#2980b9", linestyle=":", alpha=0.4, label="Act. acc threshold")
axes[1].set_xlabel("Run (best to worst)")
axes[1].set_ylabel("Metric Value")
axes[1].set_title("Metric Degradation Across Runs")
axes[1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
axes[1].set_xticks(range(n_runs))
axes[1].set_xticklabels([f"R{i}" for i in range(n_runs)])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("governance_monograph/notebooks/ch15_gate_heatmap.png", dpi=150,
            bbox_inches="tight")
plt.show()
print("Figure saved: ch15_gate_heatmap.png")

Figure saved: ch15_gate_heatmap.png


---
## Lab 15.2 --- Detect Drift with SPC

We simulate 10 agent runs with progressively degraded data. For each run we:
1. Load the Iris dataset and inject increasing levels of Gaussian noise into the features.
2. Train a RandomForestClassifier with `random_state=42`.
3. Compute RunMetrics (accuracy, confidence proxy, robustness drop, fairness proxy).
4. Build a RunTrend from the 10 runs and compute slopes.
5. Show an SPC alert when the accuracy slope turns negative.

In [7]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

iris = load_iris()
X_clean, y = iris.data, iris.target

# Split once with random_state=42
X_train_clean, X_test_clean, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42)

# Baseline model (no noise)
clf_base = RandomForestClassifier(n_estimators=100, random_state=42)
clf_base.fit(X_train_clean, y_train)
base_acc = accuracy_score(y_test, clf_base.predict(X_test_clean))
print(f"Baseline accuracy (no noise): {base_acc:.4f}")

# Simulate 10 runs with increasing noise
noise_levels = np.linspace(0.0, 2.0, 10)
run_metrics_list = []

print(f"\n{'Run':<5s} {'Noise':>7s} {'Accuracy':>9s} {'Conf':>7s} "
      f"{'RobDrop':>8s} {'AIR':>7s}")
print("-" * 50)

rng_spc = np.random.RandomState(42)

for i, noise_std in enumerate(noise_levels):
    # Inject Gaussian noise into training features
    noise_train = rng_spc.normal(0, noise_std, X_train_clean.shape)
    noise_test = rng_spc.normal(0, noise_std, X_test_clean.shape)
    X_train_noisy = X_train_clean + noise_train
    X_test_noisy = X_test_clean + noise_test

    # Train fresh model
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train_noisy, y_train)
    y_pred = clf.predict(X_test_noisy)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    # Confidence proxy: f1 * (1 - noise_std/3)
    conf = max(0.0, min(1.0, f1 * (1 - noise_std / 3.0)))

    # Robustness drop: how much worse than baseline
    rob_drop = max(0.0, base_acc - acc)

    # Fairness proxy: slight degradation with noise
    fair_air = max(0.5, 0.92 - 0.04 * noise_std + rng_spc.normal(0, 0.01))

    m = RunMetrics(
        accuracy=round(acc, 4),
        confidence=round(conf, 4),
        robustness_drop=round(rob_drop, 4),
        fairness_air=round(fair_air, 4),
        tool_calls=int(8 + rng_spc.randint(0, 4)),
        policy_blocks=int(rng_spc.randint(0, 2)),
    )
    run_metrics_list.append(m)

    print(f"{i:<5d} {noise_std:7.2f} {m.accuracy:9.4f} {m.confidence:7.4f} "
          f"{m.robustness_drop:8.4f} {m.fairness_air:7.4f}")

Baseline accuracy (no noise): 1.0000

Run     Noise  Accuracy    Conf  RobDrop     AIR
--------------------------------------------------
0        0.00    1.0000  1.0000   0.0000  0.9276
1        0.22    0.9667  0.8951   0.0333  0.9041
2        0.44    0.9333  0.7951   0.0667  0.8995
3        0.67    0.9333  0.7248   0.0667  0.8990
4        0.89    0.7667  0.5432   0.2333  0.8887


5        1.11    0.6333  0.4003   0.3667  0.8798
6        1.33    0.7000  0.3892   0.3000  0.8690
7        1.56    0.7667  0.3727   0.2333  0.8554
8        1.78    0.6000  0.2423   0.4000  0.8586
9        2.00    0.6000  0.1948   0.4000  0.8448


### Build RunTrend and Compute Slopes

Use `RunTrend` to track the 10 runs and compute OLS slopes for each metric. A negative accuracy slope signals degradation.

In [8]:
trend = RunTrend(metrics=run_metrics_list)

print("=== RunTrend Analysis ===\n")
fields = ["accuracy", "confidence", "robustness_drop", "fairness_air"]
alert_threshold = -0.01  # alert if slope < -0.01

print(f"{'Metric':<18s} {'Mean':>8s} {'Slope':>10s} {'Alert':>8s}")
print("-" * 48)

for f in fields:
    m_val = trend.mean(f)
    s_val = trend.slope(f)
    a_val = trend.alert(f, alert_threshold)
    flag = "** YES **" if a_val else "no"
    print(f"{f:<18s} {m_val:8.4f} {s_val:10.4f} {flag:>8s}")

print(f"\nAlert threshold: slope < {alert_threshold}")
print(f"Accuracy slope: {trend.slope('accuracy'):.4f}")
print(f"Confidence slope: {trend.slope('confidence'):.4f}")

acc_alert = trend.alert("accuracy", alert_threshold)
conf_alert = trend.alert("confidence", alert_threshold)
print(f"\nAccuracy drift alert:   {acc_alert}")
print(f"Confidence drift alert: {conf_alert}")

=== RunTrend Analysis ===

Metric                 Mean      Slope    Alert
------------------------------------------------
accuracy             0.7900    -0.0475 ** YES **
confidence           0.5557    -0.0914 ** YES **
robustness_drop      0.2100     0.0475       no
fairness_air         0.8827    -0.0084       no

Alert threshold: slope < -0.01
Accuracy slope: -0.0475
Confidence slope: -0.0914

Accuracy drift alert:   True
Confidence drift alert: True


### Gate Each Degraded Run

Check each of the 10 noise-degraded runs against all tiers to show how gate outcomes shift as data quality deteriorates.

In [9]:
print(f"{'Run':<5s} {'Noise':>7s} {'Acc':>7s} {'Advisory':>10s} "
      f"{'Recommend':>10s} {'Action':>10s}")
print("-" * 55)

spc_tier_counts = {"advisory": 0, "recommendation": 0, "action_taking": 0}

for i, m in enumerate(run_metrics_list):
    results = gate.check_all_tiers(m)
    for t in gate.TIER_ORDER:
        if results[t].passed:
            spc_tier_counts[t] += 1

    adv = "PASS" if results["advisory"].passed else "FAIL"
    rec = "PASS" if results["recommendation"].passed else "FAIL"
    act = "PASS" if results["action_taking"].passed else "FAIL"
    print(f"{i:<5d} {noise_levels[i]:7.2f} {m.accuracy:7.4f} "
          f"{adv:>10s} {rec:>10s} {act:>10s}")

print(f"\n=== Tier Pass Counts (out of {len(run_metrics_list)}) ===")
for t in gate.TIER_ORDER:
    print(f"  {t:18s}: {spc_tier_counts[t]:2d}/{len(run_metrics_list)}")

# Verify monotonicity still holds
all_mono_spc = all(gate.verify_monotonicity(m) for m in run_metrics_list)
print(f"\nMonotonicity holds for all degraded runs: {all_mono_spc}")

Run     Noise     Acc   Advisory  Recommend     Action
-------------------------------------------------------
0        0.00  1.0000       PASS       PASS       PASS
1        0.22  0.9667       PASS       PASS       PASS
2        0.44  0.9333       PASS       PASS       FAIL
3        0.67  0.9333       PASS       PASS       FAIL
4        0.89  0.7667       PASS       FAIL       FAIL
5        1.11  0.6333       FAIL       FAIL       FAIL
6        1.33  0.7000       PASS       FAIL       FAIL
7        1.56  0.7667       PASS       FAIL       FAIL
8        1.78  0.6000       FAIL       FAIL       FAIL
9        2.00  0.6000       FAIL       FAIL       FAIL

=== Tier Pass Counts (out of 10) ===
  advisory          :  7/10
  recommendation    :  4/10
  action_taking     :  2/10

Monotonicity holds for all degraded runs: True


### Plot SPC Trend Lines with Alert Threshold

Visualize the metric trends and regression lines. Highlight the region where the SPC alert fires.

In [10]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
x = np.arange(len(run_metrics_list))

plot_fields = [
    ("accuracy", "Accuracy", "#2980b9"),
    ("confidence", "Confidence", "#27ae60"),
    ("robustness_drop", "Robustness Drop", "#e74c3c"),
    ("fairness_air", "Fairness AIR", "#8e44ad"),
]

for ax, (field, label, color) in zip(axes.flat, plot_fields):
    vals = np.array([getattr(m, field) for m in run_metrics_list])
    slope = trend.slope(field)
    mean_val = trend.mean(field)
    intercept = mean_val - slope * x.mean()
    trend_line = intercept + slope * x

    ax.plot(x, vals, "o-", color=color, linewidth=2, markersize=6, label=f"{label}")
    ax.plot(x, trend_line, "--", color="gray", linewidth=1.5,
            label=f"Trend (slope={slope:.4f})")

    # Alert zone
    is_alert = trend.alert(field, alert_threshold)
    if is_alert:
        ax.axhspan(vals.min() - 0.05, vals.max() + 0.05,
                   alpha=0.08, color="red")
        ax.set_title(f"{label}  [ALERT: slope={slope:.4f}]",
                     fontweight="bold", color="red")
    else:
        ax.set_title(f"{label}  (slope={slope:.4f})")

    ax.set_xlabel("Run Index")
    ax.set_ylabel(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("SPC Trend Lines (alert threshold = slope < -0.01)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("governance_monograph/notebooks/ch15_spc_trends.png", dpi=150,
            bbox_inches="tight")
plt.show()
print("Figure saved: ch15_spc_trends.png")

Figure saved: ch15_spc_trends.png


---
## Exercises

### Exercise 15.1 --- Graduated Containment

Implement a graduated containment protocol: when the action_taking gate fails, automatically fall back to the recommendation tier. If recommendation also fails, fall back to advisory. If advisory fails, trigger an ABORTED state. Show the containment decision for each of the 10 degraded runs.

In [ ]:
# Exercise 15.1 — Graduated Containment
def graduated_containment(gate: TieredReleaseGate,
                          metrics: RunMetrics) -> str:
    """Return the highest tier that passes, or 'ABORTED'."""
    # Try tiers from most to least stringent
    for tier in reversed(gate.TIER_ORDER):
        result = gate.check(metrics, tier)
        if result.passed:
            return tier
    return "ABORTED"

print("=== Exercise 15.1: Graduated Containment ===\n")
print(f"{'Run':<5s} {'Noise':>7s} {'Accuracy':>9s} {'Decision':<18s} "
      f"{'Reason'}")
print("-" * 70)

containment_counts = {"action_taking": 0, "recommendation": 0,
                      "advisory": 0, "ABORTED": 0}

for i, m in enumerate(run_metrics_list):
    decision = graduated_containment(gate, m)
    containment_counts[decision] += 1

    # Get blocking reasons for the tier above the decision
    if decision == "ABORTED":
        reason = gate.check(m, "advisory").blocking_reasons[0]
    elif decision != "action_taking":
        idx = gate.TIER_ORDER.index(decision)
        next_tier = gate.TIER_ORDER[idx + 1]
        reason = gate.check(m, next_tier).blocking_reasons[0]
    else:
        reason = "all checks passed"

    print(f"{i:<5d} {noise_levels[i]:7.2f} {m.accuracy:9.4f} "
          f"{decision:<18s} {reason}")

print("\n=== Containment Summary ===")
for level, count in containment_counts.items():
    print(f"  {level:18s}: {count:2d} runs")

### Exercise 15.2 --- Cost Tracker

Add a cost ceiling to the release gate. The action_taking tier requires `tool_calls * estimated_time <= cost_ceiling`. Show that a run with many tool calls is blocked even if its metrics are good.

In [ ]:
# Exercise 15.2 — Cost Tracker
ESTIMATED_TIME_PER_CALL = 2.5  # seconds
COST_CEILING = 25.0            # max seconds for action_taking tier

def check_with_cost(gate: TieredReleaseGate, metrics: RunMetrics,
                    cost_ceiling: float = COST_CEILING) -> dict:
    """Extend gate check with a cost constraint on action_taking."""
    results = gate.check_all_tiers(metrics)

    # Add cost check to action_taking
    estimated_cost = metrics.tool_calls * ESTIMATED_TIME_PER_CALL
    cost_ok = estimated_cost <= cost_ceiling
    act_result = results["action_taking"]
    act_result.checks["cost"] = {
        "value": estimated_cost,
        "threshold": cost_ceiling,
        "passed": cost_ok,
    }
    if not cost_ok:
        act_result.blocking_reasons.append(
            f"cost {estimated_cost:.1f}s > ceiling {cost_ceiling:.1f}s")
        act_result.passed = False

    return results

print("=== Exercise 15.2: Cost Tracker ===\n")
print(f"Cost ceiling: {COST_CEILING}s "
      f"(max {int(COST_CEILING / ESTIMATED_TIME_PER_CALL)} tool calls)\n")

# Test with a high-quality but expensive run
expensive_run = RunMetrics(
    accuracy=0.95, confidence=0.85, robustness_drop=0.02,
    fairness_air=0.90, tool_calls=15, policy_blocks=0,
)
print(f"Expensive run: accuracy={expensive_run.accuracy}, "
      f"tool_calls={expensive_run.tool_calls}")
print(f"Estimated cost: {expensive_run.tool_calls * ESTIMATED_TIME_PER_CALL:.1f}s\n")

results = check_with_cost(gate, expensive_run)
for tier in gate.TIER_ORDER:
    print(gate.format_gate_result(results[tier]))
    print()

# Compare with a cheap run
cheap_run = RunMetrics(
    accuracy=0.95, confidence=0.85, robustness_drop=0.02,
    fairness_air=0.90, tool_calls=8, policy_blocks=0,
)
print(f"Cheap run: accuracy={cheap_run.accuracy}, "
      f"tool_calls={cheap_run.tool_calls}")
print(f"Estimated cost: {cheap_run.tool_calls * ESTIMATED_TIME_PER_CALL:.1f}s\n")

results_cheap = check_with_cost(gate, cheap_run)
act_cheap = results_cheap["action_taking"]
print(f"Action-taking: {'PASS' if act_cheap.passed else 'FAIL'}")
if act_cheap.blocking_reasons:
    print(f"  Blocking: {'; '.join(act_cheap.blocking_reasons)}")
else:
    print("  All checks passed (including cost).")

### Exercise 15.3 --- Adult Income Fairness Gate

Design a custom gate for the Adult Income use case. The action_taking tier requires `fairness_air >= 0.80` for both sex and race. Show that a model with high accuracy but low fairness is blocked at the action_taking tier but passes advisory.

In [ ]:
# Exercise 15.3 — Adult Income Fairness Gate
print("=== Exercise 15.3: Adult Income Fairness Gate ===\n")

# Scenario: high-accuracy model with poor fairness
# (e.g., discriminates on sex attribute)
unfair_model = RunMetrics(
    accuracy=0.88,        # good accuracy
    confidence=0.75,      # good confidence
    robustness_drop=0.03, # good robustness
    fairness_air=0.65,    # bad: AIR for sex < 0.80
    tool_calls=10,
    policy_blocks=1,
)

print("Unfair model metrics:")
print(f"  accuracy={unfair_model.accuracy}, confidence={unfair_model.confidence}")
print(f"  robustness_drop={unfair_model.robustness_drop}")
print(f"  fairness_air={unfair_model.fairness_air} (sex attribute)\n")

results_unfair = gate.check_all_tiers(unfair_model)
for tier in gate.TIER_ORDER:
    status = "PASS" if results_unfair[tier].passed else "FAIL"
    print(f"  {tier:18s}: {status}")
    if results_unfair[tier].blocking_reasons:
        for reason in results_unfair[tier].blocking_reasons:
            print(f"    -> {reason}")

# The key demonstration:
assert results_unfair["advisory"].passed, "Advisory should pass"
assert results_unfair["recommendation"].passed, "Recommendation should pass"
assert not results_unfair["action_taking"].passed, "Action_taking should fail"

print(f"\nKey result: the model passes advisory and recommendation "
      f"but is BLOCKED at action_taking due to fairness_air={unfair_model.fairness_air} < 0.80.")
print("A model risk manager would require fairness remediation before autonomous deployment.")

# Fair model: same accuracy, better fairness
fair_model = RunMetrics(
    accuracy=0.86,        # slightly lower accuracy
    confidence=0.72,
    robustness_drop=0.04,
    fairness_air=0.85,    # passes AIR threshold
    tool_calls=10,
    policy_blocks=0,
)

print(f"\nFair model: accuracy={fair_model.accuracy}, "
      f"fairness_air={fair_model.fairness_air}")
results_fair = gate.check_all_tiers(fair_model)
for tier in gate.TIER_ORDER:
    status = "PASS" if results_fair[tier].passed else "FAIL"
    print(f"  {tier:18s}: {status}")

print("\nFair model passes all tiers including action_taking.")

---
## Lab 15.3 --- Calibrated Thresholds and the Three-State Co-Attestor

The thresholds shipped on `TieredReleaseGate.TIERS` are policy-declared
defaults --- author intent, not data. High-stakes deployment requires
bootstrap-calibrated thresholds whose 95% CI an auditor can read.

We calibrate against the 1200-probe Adult Income cohort, demonstrate
the three-state co-attestor (pass/flag/block), and verify
self-consistency of the calibrated gate against the cohort.

In [ ]:
os.environ.setdefault("GOVERNED_AGENT_SIGNING_KEY",
                      "lab15-notebook-key-replace-in-deployment")

from knowlytix.harness.governance import (
    calibrate_release_gate_thresholds,
    large_adult_income_run_cohort,
    policy_declared_baseline,
    SPCSlopeCoAttestor, SPCSlopeConfig,
)

# 1200 deterministic Adult Income probes
big_cohort = large_adult_income_run_cohort(n_per_tier=300, seed=42)
print(f"Cohort: {len(big_cohort)} probes")

calibrated = calibrate_release_gate_thresholds(
    big_cohort, n_bootstrap=200, seed=42,
)
declared = policy_declared_baseline()

print("\nCalibrated thresholds (vs policy-declared):")
for tier in gate.TIER_ORDER:
    cset = calibrated[tier]
    bd = declared[tier]
    line_parts = [f"  {tier:14s}"]
    for cell_name, cell, fb in [
        ("acc",  cset.min_accuracy,        bd.min_accuracy),
        ("conf", cset.min_confidence,      bd.min_confidence),
        ("rob",  cset.max_robustness_drop, bd.max_robustness_drop),
        ("air",  cset.min_fairness_air,    bd.min_fairness_air),
    ]:
        if cell is not None:
            line_parts.append(
                f"{cell_name}={cell.mean:.4f} (decl {fb:.2f}; "
                f"CI [{cell.ci_low:.4f},{cell.ci_hi:.4f}])"
            )
        else:
            line_parts.append(f"{cell_name}=signal_unavailable")
    print(" | ".join(line_parts))

In [ ]:
# Three-state co-attestor: pass / flag / block
attestor = SPCSlopeCoAttestor(gate, config=SPCSlopeConfig())

# Case 1: clean run, no trend -> pass
clean = RunMetrics(0.92, 0.80, 0.025, 0.91, 12, 1)
v1 = attestor.evaluate(clean, "action_taking")
print(f"clean run, no trend: verdict={v1.verdict}")

# Case 2: structural fail -> block
bad = RunMetrics(0.70, 0.40, 0.08, 0.70, 12, 1)
v2 = attestor.evaluate(bad, "action_taking")
print(f"structural fail:     verdict={v2.verdict} reasons={v2.blocking_reasons[:1]}")

# Case 3: structural ok + downward slope -> flag
trend = RunTrend(metrics=[
    RunMetrics(0.92 - 0.01*i, 0.80, 0.03, 0.85, 12, 1) for i in range(8)
])
borderline = RunMetrics(0.86, 0.75, 0.04, 0.85, 12, 1)
v3 = attestor.evaluate(borderline, "action_taking", trend=trend)
print(f"borderline + drift:  verdict={v3.verdict} flag_reasons={v3.flag_reasons[:1]}")

---
## Lab 15.4 --- Audit-Defensible Hardening

Bring the release-gate surface to the same evidence base as Ch.10
(orchestrator) and Ch.12 (policy engine): bundle build with HMAC
signing, OOD validation, adversarial sweep, naive-baseline contrast,
exhaustive monotonicity proof, replay regression, and audit log.

In [ ]:
from knowlytix.harness.governance import (
    build_release_gate_bundle, bind_release_gate_from_bundle,
    ReleaseGateOperatingPoint, validate_release_gate,
    relabel_cohort_against_gate,
    build_release_gate_adversarial_cohort, sweep_release_gate_adversarial,
    FixedThresholdBaseline, RandomBaseline, compare_to_release_gate,
    verify_release_gate_invariants,
    replay_release_gate_battery,
    to_release_gate_audit_record, replay_release_gate_audit_log,
)

# Build calibrated bundle
bundle = build_release_gate_bundle(
    cohort=big_cohort, source_dataset="adult_income",
    n_bootstrap=200, seed=42,
    operating_point=ReleaseGateOperatingPoint(
        min_adversarial_safety_rate=0.95,
        min_ood_decision_match_rate=0.85,
        min_uplift_over_fixed=100,
    ),
    monotonicity_resolution=0.10,
)
print(f"bundle_id            = {bundle.bundle_id}")
print(f"cohort_hash[:16]     = {bundle.cohort_hash[:16]}")
print(f"monotonicity_hash[:16]= {bundle.monotonicity_proof_hash[:16]}")
print(f"signing_method       = {bundle.signing_method}")

# Bind calibrated gate
calibrated_gate = bind_release_gate_from_bundle(bundle, cohort=big_cohort)
print(f"\nthreshold_source     = {calibrated_gate.threshold_source}")
for tier, t in calibrated_gate.TIERS_RUNTIME.items():
    print(f"  {tier:14s}: acc={t.min_accuracy:.4f} conf={t.min_confidence:.4f} "
          f"rob={t.max_robustness_drop:.4f} air={t.min_fairness_air:.4f}")

In [ ]:
# OOD validation: both truth bases
ood_policy = validate_release_gate(calibrated_gate, big_cohort)
ood_calib = validate_release_gate(
    calibrated_gate,
    relabel_cohort_against_gate(calibrated_gate, big_cohort),
)
print(f"OOD vs policy-truth:    matches={ood_policy.decision_matches}/{ood_policy.n}")
for tier, m in ood_policy.per_tier.items():
    print(f"  {tier:14s}: FAR={m['far']:.4f} FRR={m['frr']:.4f} ECE={m['ece']:.4f}")
print(f"OOD vs calibrated-truth: matches={ood_calib.decision_matches}/{ood_calib.n}  (self-consistency)")

# Adversarial sweep
adv = build_release_gate_adversarial_cohort(n_per_class=20, seed=42, gate=calibrated_gate)
report = sweep_release_gate_adversarial(calibrated_gate, adv)
print(f"\nAdversarial: {report.n_safe}/{report.n_probes} = {report.safety_rate:.4f}")
for cls, m in sorted(report.per_class.items()):
    print(f"  {cls:42s}: {int(m['n_safe']):2d}/{int(m['n']):2d}")

In [ ]:
# Naive baselines + invariants + replay battery
fixed = FixedThresholdBaseline(min_accuracy=0.80)
random_b = RandomBaseline(seed=42)
c1 = compare_to_release_gate(calibrated_gate, big_cohort, fixed)
c2 = compare_to_release_gate(calibrated_gate, big_cohort, random_b)
print(f"vs FixedThreshold: gate={c1.n_correct_gate}, baseline={c1.n_correct_baseline}, uplift=+{c1.uplift}")
print(f"vs Random        : gate={c2.n_correct_gate}, baseline={c2.n_correct_baseline}, uplift=+{c2.uplift}")

# Exhaustive monotonicity proof
inv = verify_release_gate_invariants(calibrated_gate, resolution=0.10)
print(f"\nInvariants: {inv.n_pass}/{inv.n_grid_points} pass; {inv.n_violations} violations")
for name in inv.invariants_checked:
    print(f"  [OK] {name}")

# Replay battery
r1 = replay_release_gate_battery(calibrated_gate, big_cohort, bundle_id=bundle.bundle_id)
r2 = replay_release_gate_battery(calibrated_gate, big_cohort, bundle_id=bundle.bundle_id)
print(f"\nReplay battery: hash[:16]={r1.snapshots_hash[:16]}; byte-identical={r1.snapshots_hash == r2.snapshots_hash}")

In [ ]:
# Audit log replayable from log alone
audit_record = to_release_gate_audit_record(
    clean, "action_taking", calibrated_gate,
    co_attestor_verdict=v1, bundle=bundle,
)
verdict = replay_release_gate_audit_log(audit_record.to_dict())
print(f"audit verdict from log alone: {verdict}")

# Demonstrate domain-finiteness check (Risk fixes 3 + 4)
ok, viol = clean.is_well_formed()
print(f"\nclean run is_well_formed: {ok}")

bad_air = RunMetrics(0.90, 0.80, 0.03, 1.25, 12, 1)  # AIR > 1.0 (flipped)
ok_bad, viol_bad = bad_air.is_well_formed()
print(f"AIR=1.25 is_well_formed: {ok_bad}; violations={viol_bad}")

nan_acc = RunMetrics(float("nan"), 0.80, 0.03, 0.85, 12, 1)
ok_nan, viol_nan = nan_acc.is_well_formed()
print(f"NaN accuracy is_well_formed: {ok_nan}; violations={viol_nan[:1]}")

---
## Lab 15.5 --- Production Hardening

Independent alt-validator agreement check, latency benchmarks, and
drift simulation. The end-to-end runbook
(`gms_agent_builder_ext/lab15_release_gate_runbook.py`) executes the
same flow as a single integration script.

In [ ]:
from knowlytix.harness.governance import (
    adult_income_labeled_run_cohort,
    assert_gate_alt_agreement,
    bench_check_all_tiers, bench_co_attestor, bench_audit_record,
)

# Alt-validator agreement on cohort + adversarial probes
small = adult_income_labeled_run_cohort()
alt = assert_gate_alt_agreement(
    calibrated_gate, small + big_cohort[:200], raise_on_disagreement=False,
)
print(f"alt-validator agreement: {alt.n_agree}/{alt.n} = {alt.agreement_rate:.4f}")

# Latency benchmarks
b1 = bench_check_all_tiers(calibrated_gate, n=1000)
b2 = bench_co_attestor(calibrated_gate, n=1000)
b3 = bench_audit_record(calibrated_gate, n=1000)
print(f"\nlatency check_all_tiers: p50={b1.p50_us:5.1f}us p95={b1.p95_us:5.1f}us p99={b1.p99_us:5.1f}us  SLO {int(b1.slo_us)}us")
print(f"latency co_attestor    : p50={b2.p50_us:5.1f}us p95={b2.p95_us:5.1f}us p99={b2.p99_us:5.1f}us  SLO {int(b2.slo_us)}us")
print(f"latency audit_record   : p50={b3.p50_us:5.1f}us p95={b3.p95_us:5.1f}us p99={b3.p99_us:5.1f}us  SLO {int(b3.slo_us)}us")